<a href="https://colab.research.google.com/github/xidoudou/ai-agents/blob/main/rag01_it_helpdesk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install & Import**

In [ ]:
!pip install -q sentence-transformers openai

import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
client = OpenAI(api_key = userdata.get("OPENAI_API_KEY"))

**Knowledge Base**

In [ ]:
documents = [
    "To reset your password, go to portal.nordwerk.de and click 'Forgot password'. Passwords must be at least 12 characters long.",
    "VPN access requires a company laptop and the GlobalProtect client. If the client shows 'Gateway unreachable', contact IT.",
    "Employees may work remotely up to 3 days per week after completing their probation period.",
    "New software must be requested via the IT Service Portal. Licenses costing more than 100 EUR require line manager approval.",
    "The IT helpdesk is available Monday to Friday, 8:00 to 18:00, at phone extension 4400.",
    "Lost or stolen laptops must be reported to IT Security within 2 hours so the device can be remotely wiped. IT Security is reachable 24/7 at +49 0000 0000 000.",
]

**Ingestion (Embedding)**

In [ ]:
embeddings = embed_model.encode(documents)

**Retrieval**

In [ ]:
def retrieve(query, k = 3):
  query_embedding = embed_model.encode(query)
  scores = embed_model.similarity(query_embedding, embeddings)[0]
  top_indices = np.argsort(scores.numpy())[::-1][:k]
  return [(i, documents[i], float(scores[i])) for i in top_indices]

In [ ]:
for i, doc, score in retrieve("How to reset my password?"):

  print(f"[doc{i} score = {score:.3f} {doc[:100]}]")

**Generation**

In [ ]:
def generate_rag_response(query, k =3):
  results = retrieve(query, k)

  context = "\n".join(f"[doc{i}] {doc}" for i, doc, score in results)

  prompt = f"""
  You are a knowledge base. Please answer users' Questions according to the following rules:
  1. Only answer the content which are included in following Context, don't generate any answer using your own knowledge.
  2. If you don't find any answer of the question from Context, please reply "I don't find any information to this query"
  3. At the end of the answer please indicate the source of your answer, e.g., (Source: doc 5)

  Context:
  {context}

  Question: {query}
  """


  response = client.responses.create(
      model = "gpt-6-astra",
      input = prompt,

  )


  return response.output_text, results

**Test**

In [ ]:
answer, results = generate_rag_response("How to connect to VPN?")
print(answer)